# 04 · Validate — epitope choice, receptor-family specificity, developability

**Standard slot:** *validate (in silico).* **For Project 17 the core comparisons are:** (1) the
**epitope choice** (overlapping vs non-overlapping with an approved mAb), (2) **specificity within the
receptor family** (does the VHH prefer HER2 over EGFR/HER3/HER4?), and (3) **developability/humanness**
distributions of the survivors (D3 pt 2).

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC** — the figures here
demonstrate the analysis; real numbers come from the A100 campaign.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Epitope choice — overlapping vs non-overlapping with an approved mAb `[core]`

Re-run the campaign for **both** epitope strategies and compare the pools. The question is not "which
binds better" (mock can't answer that) but **what each strategy buys you**: an overlapping epitope
competes with / mimics a validated therapeutic site; a non-overlapping one enables biparatopic /
bispecific constructs and dodges resistance tied to the mAb site. Use `epitope_overlap()` against the
approved-mAb footprint to bin each design as competing vs orthogonal.

In [ ]:
import pandas as pd
from antibody_tools import design_vhh_cdrs, score_designs, epitope_overlap, DEFAULT_FRAMEWORK

TAA = "HER2"
FRAMEWORK = DEFAULT_FRAMEWORK
# Approved-mAb footprint (EXAMPLE — read trastuzumab's HER2 contacts off 1N8Z; verify!)
MAB_FOOTPRINT = "A557,A560,A579,A580,A583"
EPITOPES = {
    "overlapping":     "A557,A560,A579,A580,A583",   # same region as the mAb -> competes
    "non_overlapping": "A245,A266,A270,A289",         # distinct patch -> orthogonal / biparatopic
}

summary = []
for strat, ep in EPITOPES.items():
    ds = design_vhh_cdrs(TAA, ep, framework=FRAMEWORK, n=40, tool="mock")
    score_designs(ds, tool="mock")
    for d in ds:
        d.notes.append(strat)
    overlaps = [epitope_overlap(d.contact_residues, MAB_FOOTPRINT) for d in ds]
    summary.append(dict(strategy=strat, n=len(ds),
                        mean_mab_overlap=round(sum(overlaps)/len(overlaps), 3),
                        mean_pae=round(sum(d.pae_interaction for d in ds)/len(ds), 2),
                        mean_humanness=round(sum(d.humanness for d in ds)/len(ds), 3)))
epi_df = pd.DataFrame(summary)
print("Epitope-strategy comparison (SYNTHETIC mock metrics):")
epi_df

## 2 · Receptor-family specificity panel `[core]`

A TAA rarely lives alone: HER2 sits in the **HER/ErbB family** (EGFR/HER1, HER2, HER3, HER4) with
related surfaces. A useful VHH should prefer its target over the relatives. Model each survivor against
**each family member** and compare `pae_interaction`: a specific VHH has a clearly better (lower)
interface score for the target than for the off-targets. On mock this is SYNTHETIC plumbing; on Colab
run AF2-Multimer of each VHH vs each receptor (see `data/README.md` for the related-receptor panel).

In [ ]:
from antibody_tools import af2_multimer_ab

RECEPTOR_PANEL = ["HER2", "EGFR", "HER3", "HER4"]   # target + relatives (verify accessions in nb01)
TARGET = "HER2"

# Take the top few survivors from the overlapping campaign as the specificity test set.
test = design_vhh_cdrs(TARGET, EPITOPES["overlapping"], framework=FRAMEWORK, n=6, tool="mock")
score_designs(test, tool="mock")

spec_rows = []
for d in test:
    rec = {"design_id": d.design_id}
    for receptor in RECEPTOR_PANEL:
        m = af2_multimer_ab(d.sequence, antigen=receptor, tool="mock")
        rec[f"pae_{receptor}"] = m["pae_interaction"]
    # specificity margin: how much better the target is than the best off-target (lower pae = better)
    offtargets = [rec[f"pae_{r}"] for r in RECEPTOR_PANEL if r != TARGET]
    rec["spec_margin"] = round(min(offtargets) - rec[f"pae_{TARGET}"], 2)  # >0 => prefers target
    spec_rows.append(rec)
spec_df = pd.DataFrame(spec_rows)
print("Receptor-family specificity (SYNTHETIC mock pae_interaction; >0 spec_margin prefers target):")
spec_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(spec_df))
w = 0.2
for i, receptor in enumerate(RECEPTOR_PANEL):
    ax.bar(x + i*w, spec_df[f"pae_{receptor}"], width=w, label=receptor)
ax.set_xticks(x + 1.5*w); ax.set_xticklabels(spec_df["design_id"], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("pae_interaction (Å, mock)")
ax.set_title("Receptor-family specificity panel — SYNTHETIC (lower = better; want target lowest)")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("results/specificity_panel.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. On Colab, model each VHH vs each receptor with AF2-Multimer.")

## 3 · Developability + humanness figures `[core]`

Plot the survivors' **TAP-like** liability score, **CamSol-like** solubility, and **humanness** proxy.
These are **TEACHING HEURISTICS** (`antibody_tools.developability`), not the validated tools — the goal
is to teach the developability *axes* and to triage obvious liabilities (long CDR3, free cysteines,
N-glyc sequons, hydrophobic patches) **before** synthesis. Swap in real TAP/CamSol/Hu-mAb for any
reportable claim.

In [ ]:
camp = pd.read_csv("results/campaign.csv")

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].hist(camp["tap_score"], bins=12); ax[0].set_title("TAP-like liabilities (fewer better)")
ax[1].hist(camp["camsol_like"], bins=12); ax[1].set_title("CamSol-like solubility (higher better)")
ax[2].hist(camp["humanness"], bins=12); ax[2].set_title("humanness (higher better)")
plt.suptitle("Developability heuristics (NOT validated tools) — SYNTHETIC on mock")
plt.tight_layout(); plt.savefig("results/developability.png", dpi=150); plt.show()

# A simple developability triage flag (teaching): low liabilities AND humanized-enough.
camp["dev_ok"] = (camp["tap_score"] <= camp["tap_score"].median()) & (camp["humanness"] >= 0.5)
print("developability-OK (heuristic) fraction:", round(camp["dev_ok"].mean(), 3),
      "— TEACHING triage only; confirm with real TAP/CamSol/humanness tools.")

## D3 (part 2) checklist
- [ ] Epitope-choice comparison (overlapping vs non-overlapping) with what each strategy buys you.
- [ ] Receptor-family **specificity panel** (target vs relatives) + figure; specificity margin reported.
- [ ] Developability/humanness figures, flagged as **heuristics** (real tools for any claim).
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as plumbing/teaching, not results.

**Next:** `05_validation_plan.ipynb` — the display-screen plan + downstream format + controls.